In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from ECOv002_calval_tables import load_calval_table
from BESS_JPL import load_ECOv002_static_tower_BESS_inputs
from BESS_JPL import process_BESS_table

In [3]:
repo_root = os.path.dirname(os.getcwd())
package_dir = os.path.join(repo_root, 'BESS_JPL')
generated_input_table_filename = os.path.join(package_dir, "ECOv002-cal-val-BESS-JPL-inputs.csv")
generated_output_table_filename = os.path.join(package_dir, "ECOv002-cal-val-BESS-JPL-outputs.csv")

In [4]:
model_inputs_gdf = load_calval_table()
model_inputs_gdf["elevation_km"] = model_inputs_gdf["Elev"] / 1000.0
model_inputs_gdf.head()

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,LE_count,closure_ratio,geometry,time_UTC,ST_K,ST_C,Ta_C,SWin_Wm2,emissivity,elevation_km
0,0,US-NC3,ENF,Cfa,270.34520,78.53355,392.85184,307.02197,487.383423,118.91628,...,9576,1.02,POINT (-76.656 35.799),2019-10-02 19:09:40,305.10,31.95,32.658920,545.51056,0.948,0.005
1,1,US-Mi3,CVM,Dfb,232.14160,229.20093,640.11847,375.08930,106.825577,167.91946,...,12170,0.92,POINT (-80.637 41.8222),2019-06-23 18:17:17,304.34,31.19,24.227982,848.34390,0.952,0.270
2,2,US-Mi3,CVM,Dfb,356.35574,335.23154,625.66170,284.68625,NaN,132.93634,...,12170,0.92,POINT (-80.637 41.8222),2019-06-27 16:35:42,304.06,30.91,26.178862,838.81160,0.972,0.270
3,3,US-Mi3,CVM,Dfb,332.93840,326.68680,624.25433,251.41449,178.827545,141.13242,...,12170,0.92,POINT (-80.637 41.8222),2019-06-30 15:44:10,301.80,28.65,22.527096,851.72480,0.974,0.270
4,4,US-Mi3,CVM,Dfb,286.85403,237.21654,511.08218,228.52017,154.791626,114.80941,...,12170,0.92,POINT (-80.637 41.8222),2019-07-01 14:53:48,303.18,30.03,23.280691,702.55160,0.960,0.270


In [5]:
static_inputs_df = load_ECOv002_static_tower_BESS_inputs()
static_inputs_df

,ID,name,NDVI_minimum,NDVI_maximum,C4_fraction,carbon_uptake_efficiency,kn,peakVCmax_C3,peakVCmax_C4,ball_berry_slope_C3,...,KG_climate,CI,canopy_height_meters,COT,AOT,Ca,wind_speed_mps,vapor_gccm,ozone_cm,geometry
0,US-NC3,NC_Clearcut#3,0.408733,0.855693,0.077532,0.080000,0.410000,87.345433,51.040624,9.5,...,3,0.282353,20.642902,0,0,400,0,0,0.3,POINT (-76.656 35.799)
1,PE-QFR,Quistococha Forest Reserve,0.657359,0.826605,0.000549,0.060347,0.125033,41.364487,41.364487,9.5,...,1,0.254902,22.140021,0,0,400,0,0,0.3,POINT (-73.319 -3.8344)
2,US-Mi3,LTAR UCB (Upper Chesapeake Bay) Miscanthus 3,0.027910,0.855461,0.035045,0.080000,0.410000,119.435443,119.435443,7.5,...,4,0.286275,0.000000,0,0,400,0,0,0.3,POINT (-80.637 41.8222)
3,US-NC4,NC_AlligatorRiver,0.591358,0.869758,0.046219,0.080000,0.410000,64.720165,64.720165,9.5,...,3,0.207843,14.164827,0,0,400,0,0,0.3,POINT (-75.9038 35.7879)
4,CA-DB2,Delta Burns Bog 2,0.399712,0.675731,0.000356,0.080000,0.410000,109.986395,109.986395,9.5,...,3,0.266667,9.919029,0,0,400,0,0,0.3,POINT (-122.9951 49.119)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,US-xSL,"NEON North Sterling, CO (STER)",-0.025449,0.495206,0.340776,0.090000,0.710000,78.000000,40.000000,9.5,...,2,0.298039,0.000000,0,0,400,0,0,0.3,POINT (-103.0293 40.4619)
117,US-xWD,NEON Woodworth (WOOD),-0.041785,0.753863,0.032479,0.080000,0.410000,101.000000,37.000000,7.5,...,4,0.294118,0.000000,0,0,400,0,0,0.3,POINT (-99.2414 47.1282)
118,US-CS4,Central Sands Irrigated Agricultural Field,-0.003026,0.776740,0.092454,0.080000,0.410000,101.000000,37.000000,7.5,...,4,0.278431,0.000000,0,0,400,0,0,0.3,POINT (-89.5475 44.1597)
119,US-xAE,NEON Klemme Range Research Station (OAES),0.233503,0.554538,0.371127,0.090000,0.710000,78.000000,40.000000,9.5,...,3,0.301961,0.000000,0,0,400,0,0,0.3,POINT (-99.0588 35.4106)


In [6]:
# merge static inputs with model inputs, ignoring duplicate columns from static_inputs_df
cols_to_use = [col for col in static_inputs_df.columns if col not in model_inputs_gdf.columns or col == 'ID']
model_inputs_gdf = model_inputs_gdf.merge(static_inputs_df[cols_to_use], on="ID", how="left")
model_inputs_gdf

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,ball_berry_intercept_C3,KG_climate,CI,canopy_height_meters,COT,AOT,Ca,wind_speed_mps,vapor_gccm,ozone_cm
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0.005267,3,0.282353,20.642902,0,0,400,0,0,0.3
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0.009882,4,0.286275,0.000000,0,0,400,0,0,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0.015000,3,0.301961,0.000000,0,0,400,0,0,0.3


In [7]:
model_inputs_gdf.columns

Index(['Unnamed: 0', 'ID', 'vegetation', 'climate', 'STICinst', 'BESSinst',
       'MOD16inst', 'PTJPLSMinst', 'ETinst', 'ETinstUncertainty', 'PET', 'Rn',
       'ESI', 'RH', 'Ta', 'LST', 'SM', 'NDVI', 'NDVI-UQ', 'albedo',
       'albedo-UQ', 'LST_err', 'view_zenith', 'Rg', 'EmisWB', 'time_utc',
       'solar_time', 'solar_hour', 'local_time', 'LE', 'LE_filt', 'LEcorr25',
       'LEcorr50', 'LEcorr75', 'LEcorr_ann', 'H_filt', 'Hcorr25', 'Hcorr50',
       'Hcorr75', 'Hcorr_ann', 'NETRAD_filt', 'G_filt', 'SM_surf', 'SM_rz',
       'AirTempC', 'SW_IN', 'RH_percentage', 'ESIrn_STIC', 'ESIrn_PTJPLSM',
       'ESIrn_MOD16', 'ESIrn_BESS', 'ESIrn_Unc_ECO', 'ESIrn_LEcorr50', 'JET',
       'eco_time_utc', 'Site Name', 'Date-Time', 'Site ID', 'Name', 'Lat',
       'Long', 'Elev', 'Clim', 'Veg', 'MAT', 'MAP', 'StartDate', 'EndDate',
       'LE_count', 'closure_ratio', 'geometry', 'time_UTC', 'ST_K', 'ST_C',
       'Ta_C', 'SWin_Wm2', 'emissivity', 'elevation_km', 'name',
       'NDVI_minimum', 'ND

In [8]:
results = process_BESS_table(model_inputs_gdf)
results

[2025-09-09 16:54:04 INFO] started extracting geometry from PT-JPL-SM input table
[2025-09-09 16:54:04 INFO] completed extracting geometry from PT-JPL-SM input table
[2025-09-09 16:54:04 INFO] started extracting time from PT-JPL-SM input table
[2025-09-09 16:54:04 INFO] completed extracting time from PT-JPL-SM input table
[2025-09-09 16:54:04 INFO] GEOS-5 FP working directory: /Users/gregoryhalverson/data/GEOS5FP
[2025-09-09 16:54:04 INFO] GEOS-5 FP download directory: ~/data/GEOS5FP
[2025-09-09 16:54:04 INFO] variable elevation_km min: 0.001 mean: 0.993 max: 3.504 nan: 0.00% (nan)
[2025-09-09 16:54:04 INFO] variable Ta_C min: -14.605 mean: 22.322 max: 39.710 nan: 0.00% (nan)
[2025-09-09 16:54:04 INFO] variable RH min: 0.273 mean: 0.427 max: 0.984 nan: 0.00% (nan)
[2025-09-09 16:54:04 INFO] variable NDVI_minimum min: -0.033 mean: 0.174 max: 0.591 nan: 0.00% (nan)
[2025-09-09 16:54:04 INFO] variable NDVI_maximum min: 0.314 mean: 0.628 max: 0.918 nan: 0.00% (nan)
[2025-09-09 16:54:04 INF

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,wind_speed_mps,vapor_gccm,ozone_cm,GPP,GPP_daily,Rn_soil,Rn_canopy,LE_soil,LE_canopy,G
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0,0,0.3,16.306166,5.963702,343.088141,200.486306,69.342528,184.004129,37.045927
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0,0,0.3,21.081704,8.623039,568.929428,176.618973,100.628336,125.136978,75.785976
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0,0,0.3,19.984935,8.120491,595.207535,201.872465,210.893323,135.914517,77.675719
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0,0,0.3,24.216487,10.357271,539.752171,195.049305,207.202548,129.500884,69.717555
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0,0,0.3,21.872922,10.242349,441.941830,158.185783,140.032444,119.645512,57.794590
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0,0,0.3,0.492826,0.193133,165.684047,69.659646,41.487784,5.193484,29.244793
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0,0,0.3,1.259612,0.882347,190.262323,75.928297,31.172926,11.745809,32.896598
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0,0,0.3,3.360783,2.404805,266.716370,138.943197,6.239409,41.129234,35.811898
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0,0,0.3,1.986428,1.197719,247.995207,66.902355,48.996907,10.137635,44.868324


In [9]:
model_inputs_gdf.to_csv(generated_input_table_filename, index=False)

In [10]:
results.to_csv(generated_output_table_filename, index=False)